# Mini-GPT — Phase 1: Baseline Model

**ES532 Deep Learning course project.** This notebook is Phase 1 of the plan: a plain,
correct, FP32 baseline GPT-style transformer, built from scratch.

Later phases (mixed precision, gradient checkpointing, a custom fused Triton kernel,
KV-cache, quantization) will each get their own notebook/section that reuses this same
model code unchanged — the whole point of the project is that the model doesn't change,
only how efficiently it trains/runs.

**Before running:** in the Colab menu, go to `Runtime -> Change runtime type -> T4 GPU`,
so the GPU check below actually finds a device.

## Setup

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f'PyTorch version: {torch.__version__}')

In [ ]:
# Confirm the T4 GPU is actually attached before doing anything else.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU found — go to Runtime > Change runtime type > T4 GPU, then re-run this cell.')

## Config

All architecture hyperparameters live here. Keep this small enough to train comfortably
on a single T4 within a Colab session (default below is ~a few million params for quick
testing — we'll size it up to ~15-30M once real data is wired in).

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 5000       # set from your tokenizer once data prep exists
    context_length: int = 256    # max sequence length the model can attend over
    n_layers: int = 6
    n_heads: int = 6
    n_embd: int = 384            # embedding dimension (must be divisible by n_heads)
    dropout: float = 0.1
    bias: bool = True            # whether Linear/LayerNorm layers have a bias term

    def __post_init__(self):
        assert self.n_embd % self.n_heads == 0, 'n_embd must be divisible by n_heads'

## Building blocks

`LayerNorm`, `CausalSelfAttention`, `MLP`, and `Block` — the standard pieces of a
GPT-2-style decoder-only transformer, pre-norm style.

In [ ]:
class LayerNorm(nn.Module):
    """LayerNorm with an optional bias (PyTorch's built-in doesn't let you turn
    bias off directly). Kept as its own small module now so Phase 4 can later
    swap it for a fused Triton kernel without touching anything else."""

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, eps=1e-5)

In [ ]:
class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention -- the vanilla (unfused) version.

    Phase 4 introduces model/attention.py with a Triton-fused alternative; at
    that point this class becomes the baseline we benchmark the fused kernel
    against, so don't delete it later -- just add the alternative alongside it.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_heads

        # one Linear projects to Q, K, V all at once (fewer kernel launches than 3 separate ones)
        self.qkv_proj = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.resid_dropout = nn.Dropout(config.dropout)
        self.dropout = config.dropout

    def forward(self, x):
        B, T, C = x.shape  # batch, sequence length, embedding dim

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # reshape to (B, n_heads, T, head_dim) so attention is computed per head
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # scaled_dot_product_attention already dispatches to a fused, memory-efficient
        # kernel under the hood (Flash Attention on supported GPUs incl. the T4) --
        # this IS our real Phase-4 comparison target once we hand-write our own.
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,   # token i can only attend to tokens <= i
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        return y

In [ ]:
class MLP(nn.Module):
    """The feed-forward block inside each transformer layer. Standard 4x expansion."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.fc_in = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.fc_out = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc_in(x)
        x = self.gelu(x)
        x = self.fc_out(x)
        x = self.dropout(x)
        return x

In [ ]:
class Block(nn.Module):
    """One transformer block: pre-norm attention + pre-norm MLP, each with a
    residual connection. "Pre-norm" = LayerNorm happens BEFORE attention/MLP,
    the modern, more stable convention used from GPT-2 onward."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

## The full model

Assembles the blocks above into `MiniGPT`, with weight tying between the input
embedding and output head (a standard GPT-2 trick).

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.context_length, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.ln_f = LayerNorm(config.n_embd, bias=config.bias)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # weight tying: input embedding and output projection share the same weights.
        # Standard GPT-2 trick -- saves params and tends to help quality a little.
        self.token_embedding.weight = self.lm_head.weight

        self.apply(self._init_weights)
        print(f'MiniGPT initialized: {self.num_params() / 1e6:.2f}M parameters')

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None):
        """
        idx:     (B, T) tensor of token ids
        targets: (B, T) tensor of token ids to predict (idx shifted by 1), or None at inference

        Returns (logits, loss). loss is None if targets is None.
        """
        B, T = idx.shape
        assert T <= self.config.context_length, (
            f'sequence length {T} exceeds context_length {self.config.context_length}'
        )

        positions = torch.arange(0, T, dtype=torch.long, device=idx.device)

        tok_emb = self.token_embedding(idx)            # (B, T, n_embd)
        pos_emb = self.position_embedding(positions)    # (T, n_embd), broadcasts over batch
        x = self.dropout(tok_emb + pos_emb)

        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        NAIVE generation (the Phase 1-4 baseline) -- recomputes the full forward
        pass over the whole sequence for every new token. Intentionally simple
        and slow: generate.py will add a KV-cached version for Phase 5, and the
        whole point of that phase is to benchmark it against this one.

        idx: (B, T) tensor of token ids to start from (the "prompt")
        """
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.context_length \
                else idx[:, -self.config.context_length:]

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # only need the last token's logits

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

## Sanity check

Builds a tiny model, runs one forward + backward pass, and generates a few tokens —
just to confirm everything is wired correctly before touching real data. Runs on the
T4 if one is attached, otherwise falls back to CPU (slower, but fine for this quick check).

In [ ]:
config = GPTConfig(vocab_size=1000, context_length=64, n_layers=4, n_heads=4, n_embd=128)
model = MiniGPT(config).to(device)

dummy_input = torch.randint(0, config.vocab_size, (2, 32), device=device)    # batch=2, seq_len=32
dummy_targets = torch.randint(0, config.vocab_size, (2, 32), device=device)

logits, loss = model(dummy_input, dummy_targets)
print(f'logits shape: {logits.shape}')   # expect (2, 32, 1000)
print(f'loss: {loss.item():.4f}')

loss.backward()
print('Backward pass succeeded — model is wired up correctly.')

generated = model.generate(dummy_input[:, :5], max_new_tokens=10)
print(f'generated shape: {generated.shape}')   # expect (2, 15)

## Next up

Next file: `data/prepare_data.py` — downloading and tokenizing a real text dataset
(TinyShakespeare or WikiText-2) so Phase 1 training can run on real data instead of
random dummy tensors.